In [10]:
import pyautogui
import pandas as pd
import numpy as np
from importlib import reload
from source import select


## Read in telemetry file

### **select file**

In [2]:
reload(select)

ifile = select.single_file()

ifile

'C:/Users/CTD/ADCP/adcp-py-toolbox/data/txt/k01_c_deploy.txt'

### **read data into dictionary**

In [3]:
import re
from collections import defaultdict

data = defaultdict(lambda: defaultdict(list))

with open(ifile, 'r') as f:
    for line in f:
        try:
            match = re.match(r'\$(\w+),(.*)\*(\w+)', line.strip())
            if not match:
                continue
            tm_type, content = match.group(1), match.group(2)
            fields = {}
            for field in content.split(','):
                if '=' in field:
                    k, v = field.split('=', 1)
                    fields[k] = v
            for key, value in fields.items():
                data[tm_type][key].append(value)
        except Exception as e:
            print(f"Skipping line due to error: {line.strip()}\n{e}")

data.keys()

dict_keys(['PNORI2', 'PNORS2', 'PNORC2'])

### **check contents**

In [4]:
pd.DataFrame(data['PNORC2'])

,DATE,TIME,CN,CP,VE,VN,VU,VU2,A1,A2,A3,A4,C1,C2,C3,C4
0,070325,183345,1,0.3,-0.077,-0.012,-0.031,-0.021,84.1,83.9,84.0,84.1,97,89,95,100
1,070325,183345,2,0.5,0.045,0.010,-0.040,-0.002,81.3,81.3,81.9,81.4,86,83,91,88
2,070325,183345,3,0.7,0.050,-0.051,-0.005,0.016,78.9,81.1,79.9,79.1,98,100,99,97
3,070325,183345,4,0.9,0.113,-0.063,0.028,0.004,78.3,79.4,77.3,77.0,95,74,97,96
4,070325,183345,5,1.1,0.025,0.007,0.030,0.014,76.3,74.8,75.4,76.1,97,89,100,91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13195,070425,091145,26,5.3,0.063,-0.013,0.000,-0.006,58.9,58.4,60.1,59.0,96,98,99,98
13196,070425,091145,27,5.5,-0.041,0.006,-0.001,0.004,60.9,57.0,59.5,59.3,98,97,93,98
13197,070425,091145,28,5.7,0.079,0.043,-0.001,0.001,60.4,57.0,60.5,58.4,92,100,99,96
13198,070425,091145,29,5.9,0.014,0.120,0.041,-0.013,57.5,57.2,57.8,57.8,94,97,87,99


## Save and format data

### **convert to xarray.Dataset**

In [5]:
import xarray as xr
import pandas as pd

def parse_time(df):
    df['TIME'] = pd.to_datetime(df['DATE']+' '+df['TIME'], format='%m%d%y %H%M%S')
    return df.drop(columns='DATE')

try:
    # Process PNORC2
    cdf = parse_time(pd.DataFrame(data['PNORC2'])).set_index(['TIME', 'CN'])
    ds = cdf.to_xarray()
    
    # Process PNORS2
    sdf = parse_time(pd.DataFrame(data['PNORS2'])).set_index('TIME')
    for col in sdf.columns:
        ds[col] = ('TIME', sdf[col])
    
    # Add PNORI2
    for k, v in data['PNORI2'].items():
        ds[k] = v[0]

except KeyError as e:
    print(f"Missing expected keys in data: {e}")
    ds = xr.Dataset()

ds

<xarray.Dataset> Size: 1MB
Dimensions:  (TIME: 440, CN: 30)
Coordinates:
  * TIME     (TIME) datetime64[us] 4kB 2025-07-03T18:33:45 ... 2025-07-04T09:...
  * CN       (CN) object 240B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
Data variables: (12/33)
    CP       (TIME, CN) object 106kB '0.3' '2.1' '2.3' ... '1.5' '1.7' '1.9'
    VE       (TIME, CN) object 106kB '-0.077' '-0.120' ... '0.001' '0.023'
    VN       (TIME, CN) object 106kB '-0.012' '0.035' ... '-0.057' '0.035'
    VU       (TIME, CN) object 106kB '-0.031' '0.015' ... '0.027' '0.012'
    VU2      (TIME, CN) object 106kB '-0.021' '-0.025' ... '-0.040' '0.013'
    A1       (TIME, CN) object 106kB '84.1' '67.1' '68.2' ... '71.1' '71.7'
    ...       ...
    SN       <U6 24B '106516'
    NB       <U1 4B '4'
    NC       <U2 8B '30'
    BD       <U4 16B '0.10'
    CS       <U4 16B '0.20'
    CY       <U3 12B 'ENU'

In [7]:
import numpy as np

def assign_dtypes(ds, dtypes, fill=None):
    for col in ds.variables:
        if col in dtypes.keys():
            ds[col] = ds[col].astype(dtypes[col])
        else:
            if fill: 
                ds[col] = ds[col].astype(fill)

dtypes = dict(
    TIME = np.datetime64,
    CN = int,
    EC = int,
    IT = int,
    NB = int,
    NC = int,
    SC = str,
    SN = str,
    CY = str
)

assign_dtypes(ds, dtypes, fill=float)

ds

<xarray.Dataset> Size: 1MB
Dimensions:  (TIME: 440, CN: 30)
Coordinates:
  * TIME     (TIME) datetime64[us] 4kB 2025-07-03T18:33:45 ... 2025-07-04T09:...
  * CN       (CN) int64 240B 1 10 11 12 13 14 15 16 17 ... 29 3 30 4 5 6 7 8 9
Data variables: (12/33)
    CP       (TIME, CN) float64 106kB 0.3 2.1 2.3 2.5 2.7 ... 1.3 1.5 1.7 1.9
    VE       (TIME, CN) float64 106kB -0.077 -0.12 0.06 ... -0.052 0.001 0.023
    VN       (TIME, CN) float64 106kB -0.012 0.035 -0.024 ... 0.018 -0.057 0.035
    VU       (TIME, CN) float64 106kB -0.031 0.015 0.014 ... -0.059 0.027 0.012
    VU2      (TIME, CN) float64 106kB -0.021 -0.025 -0.025 ... -0.04 0.013
    A1       (TIME, CN) float64 106kB 84.1 67.1 68.2 70.9 ... 71.9 71.1 71.7
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 30
    BD       float64 8B 0.1
    CS       float64 8B 0.2
    CY       <U3 12B 'ENU'

In [9]:
print(ds)

<xarray.Dataset> Size: 1MB
Dimensions:  (TIME: 440, CN: 30)
Coordinates:
  * TIME     (TIME) datetime64[us] 4kB 2025-07-03T18:33:45 ... 2025-07-04T09:...
  * CN       (CN) int64 240B 1 10 11 12 13 14 15 16 17 ... 29 3 30 4 5 6 7 8 9
Data variables: (12/33)
    CP       (TIME, CN) float64 106kB 0.3 2.1 2.3 2.5 2.7 ... 1.3 1.5 1.7 1.9
    VE       (TIME, CN) float64 106kB -0.077 -0.12 0.06 ... -0.052 0.001 0.023
    VN       (TIME, CN) float64 106kB -0.012 0.035 -0.024 ... 0.018 -0.057 0.035
    VU       (TIME, CN) float64 106kB -0.031 0.015 0.014 ... -0.059 0.027 0.012
    VU2      (TIME, CN) float64 106kB -0.021 -0.025 -0.025 ... -0.04 0.013
    A1       (TIME, CN) float64 106kB 84.1 67.1 68.2 70.9 ... 71.9 71.1 71.7
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 30
    BD       float64 8B 0.1
    CS       float64 8B 0.2
    CY       <U3 12B 'ENU'


### **export to netCDF**

In [96]:
import os

filename = os.path.basename(ifile).rpartition('.')[0]
ofile = f'data/formatted/{filename}.nc'

ds.to_netcdf(ofile)